# 05 - Train/Test Preparation and Model Development

**Phase 5: Train/Test Preparation** · Group AI-39

This notebook creates the single chronological train/test split used by all models, then trains the Phase 6 baseline and four classifiers. The final season (`2015/2016`) is held out as an untouched future test set. Historical-feature imputation values are calculated from training rows only.

In [1]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.train_models import (
    TEST_SEASON,
    tune_models,
    build_split_summary,
    chronological_split,
    load_features,
    summarize_class_distribution,
    build_training_summary,
    fit_models,
    save_predictions,
)

features_path = REPO_ROOT / "data" / "processed" / "features.csv"
features = load_features(features_path)


In [ ]:
prepared = chronological_split(features, test_season=TEST_SEASON)
tuning = tune_models(features)
tuning.tried_configs
split_summary = build_split_summary(prepared)
split_summary


In [ ]:
class_distribution = pd.concat(
    [
        summarize_class_distribution(features["match_outcome"], "overall"),
        summarize_class_distribution(prepared.y_train, "train"),
        summarize_class_distribution(prepared.y_test, "test"),
    ],
    ignore_index=True,
)
class_distribution


In [ ]:
pd.Series(prepared.imputation_values, name="training_median")


The model feature matrix contains historical numeric features and `league_id`. Identifiers, dates, season labels, and the target are retained only in the split row tables for traceability and are not passed as model features. No match IDs overlap between train and test, and the training period ends before the test period begins.

## Phase 6 - Model development

Every estimator below uses the same chronological split and feature columns. Hyperparameters are selected by macro F1 on validation season `2014/2015` using `2009/2010`–`2013/2014` as tuning training data. The final selected configurations are then refit on `2009/2010`–`2014/2015` and evaluated only on untouched `2015/2016`. Logistic Regression, Decision Tree, and Random Forest use `class_weight="balanced"`; Logistic Regression and KNN scale numeric historical features, while the tree models do not. `league_id` is one-hot encoded inside each model pipeline.

In [ ]:
trained = fit_models(prepared, configs=tuning.selected_configs)
training_summary = build_training_summary(prepared, tuning.selected_configs)
training_summary


In [ ]:
predictions_path = REPO_ROOT / "outputs" / "predictions" / "test_predictions.csv"
save_predictions(trained.predictions, predictions_path)
training_summary_path = REPO_ROOT / "outputs" / "tables" / "model_training_summary.csv"
training_summary.to_csv(training_summary_path, index=False)
print(predictions_path)
print(training_summary_path)


In [ ]:
tuning_path = REPO_ROOT / "outputs" / "tables" / "hyperparameter_tuning.csv"
tuning.tried_configs.to_csv(tuning_path, index=False)
print(tuning_path)
